# 🦺 Construction Site PPE Detection - YOLOv8 GPU Training Pipeline
### ระบบฝึกฝนโมเดล AI ตรวจจับอุปกรณ์ความปลอดภัย PPE ด้วย Google Colab (GPU เร่งความเร็ว)
---
สมุดโค้ด (Notebook) นี้ถูกออกแบบมาสำหรับเทรนโมเดลตรวจจับ PPE (หมวกนิรภัย, เสื้อกั๊ก, คน, หน้ากาก) ด้วย GPU ฟรีของ Google Colab ซึ่งเร็วกว่าการเทรนบนคอมพิวเตอร์ทั่วไป **30 - 50 เท่า!**

📌 **สิ่งที่ต้องเตรียม:**
1. บัญชี Google (เพื่อใช้ Google Colab ฟรี)
2. เปลี่ยน Runtime เป็น GPU: เมนู `Runtime` -> `Change runtime type` -> เลือก `T4 GPU`

## 1. ตรวจสอบสถานะการเชื่อมต่อ GPU ของ Google Colab
เช็คว่า Colab เชื่อมต่อกับการ์ดจอ **NVIDIA T4 (15–16 GB VRAM)** หรือไม่

In [ ]:
!nvidia-smi
import torch
print('[*] CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('[*] GPU Device Name:', torch.cuda.get_device_name(0))

## 2. ติดตั้ง Ultralytics (YOLOv8) และไลบรารีที่จำเป็น

In [ ]:
!pip install -q ultralytics roboflow openvino onnx

## 3. ดึง Dataset สำหรับเทรนเข้าสู่ Colab
*สามารถเลือก 1 ใน 2 วิธีนี้ตามความสะดวก:*

In [ ]:
# ดึง Dataset จาก Roboflow Universe โดยตรง
!pip install -q roboflow
from roboflow import Roboflow
rf = Roboflow(api_key="JZxCGXvWDAkiBZjEEx7Z")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
dataset = version.download("yolov8")
print('[+] Dataset location:', dataset.location)


## 4. กำหนดคอนฟิกไฟล์ data.yaml สำหรับ 10 คลาส PPE
คัดกรองและจับคู่วัตถุความปลอดภัย 10 คลาสหลัก

In [ ]:
import os
# ใช้ไฟล์ data.yaml ที่ Roboflow สร้างมาให้โดยอัตโนมัติ
data_yaml_path = f"{dataset.location}/data.yaml"
print('[+] Using config:', data_yaml_path)
with open(data_yaml_path, 'r') as f:
    print(f.read())


## 5. เริ่มต้นการเทรนด้วย GPU NVIDIA ความเร็วสูง (YOLOv8s)
ด้วยพลังของ GPU T4 จะใช้เวลาเพียงประมาณ 2–3 นาทีต่อ 1 Epoch เท่านั้น!

In [ ]:
from ultralytics import YOLO

# โหลดโมเดล YOLOv8s มาตรฐาน หรือ fine-tune จากไฟล์ best.pt
model = YOLO('yolov8s.pt')

results = model.train(
    data=data_yaml_path,
    epochs=50,             # จำนวนรอบที่ต้องการ (50-100 epochs บน GPU รันแป๊บเดียวเสร็จ)
    batch=32,              # รันทีละ 32 รูปด้วย VRAM ของ GPU
    imgsz=640,
    device=0,              # ใช้ GPU NVIDIA (Device 0)
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    box=7.5,
    cls=1.2,
    dfl=1.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.15,
    project='/content/runs/detect',
    name='ppe_yolov8s_colab',
    exist_ok=True,
    save=True,
    verbose=True
)

print('[+] Training Completed!')

## 6. ประเมินผลความแม่นยำบน Test Set & ดูกราฟ
ดูค่า mAP, Confusion Matrix และ F1-Score ของโมเดล

In [ ]:
from IPython.display import Image, display

# แสดงผล Confusion Matrix และกราฟการเรียนรู้
try:
    display(Image(filename='/content/runs/detect/ppe_yolov8s_colab/results.png'))
    display(Image(filename='/content/runs/detect/ppe_yolov8s_colab/confusion_matrix.png'))
except Exception as e:
    print(f'Chart display error: {e}')

# ประเมินชุด Test (ภาพที่ไม่เคยเห็นมาก่อน)
best_model = YOLO('/content/runs/detect/ppe_yolov8s_colab/weights/best.pt')
metrics = best_model.val(data=data_yaml_path, split='test')
print(f'[+] Test Set mAP@50: {metrics.box.map50*100:.2f}%')
print(f'[+] Test Set mAP@50-95: {metrics.box.map*100:.2f}%')

## 7. แปลงโมเดลเป็น OpenVINO / ONNX และดาวน์โหลดไฟล์ best.pt กลับมาใช้งาน
ดาวน์โหลดไฟล์สมองขนาด 22 MB นำไปใส่ในโปรเจกต์คอมพิวเตอร์ของคุณ

In [ ]:
from google.colab import files

# ส่งออกเป็น OpenVINO FP16 และ ONNX
best_model.export(format='openvino', half=True)
best_model.export(format='onnx', dynamic=True)

# สั่งดาวน์โหลดไฟล์ best.pt กลับเข้าเครื่องคอมพิวเตอร์ของคุณ
files.download('/content/runs/detect/ppe_yolov8s_colab/weights/best.pt')
print('[+] Click download above to save best.pt to your computer!')